In [1]:
# 
# SMOKING DETECTION - YOLOv9 TRAINING & EVALUATION
# Optimized for Kaggle (GPU)
# 

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import warnings
warnings.filterwarnings('ignore')

# Install ultralytics (supports YOLOv9)
!pip install -q ultralytics

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ============================================
# 1. LOCATE YOUR DATASET
# ============================================
possible_paths = [
"/kaggle/input/datasets/kmanivignesh/smoking-finalised-dataset/Smoking.v3i.yolov8",
    "Smoking.v3i.yolov8",
]

data_yaml = None
for p in possible_paths:
    test_yaml = Path(p) / "data.yaml"
    if test_yaml.exists():
        data_yaml = str(test_yaml)
        base_path = str(Path(p))
        print(f"✅ Dataset found at: {base_path}")
        break

if data_yaml is None:
    # Search inside /kaggle/input
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.yaml" in files:
            data_yaml = os.path.join(root, "data.yaml")
            base_path = os.path.dirname(data_yaml)
            print(f"✅ Found at: {base_path}")
            break

if data_yaml is None:
    raise FileNotFoundError("Could not find data.yaml. Make sure your dataset is uploaded correctly.")

# Verify single class (smoking)
with open(data_yaml, 'r') as f:
    import yaml
    config = yaml.safe_load(f)
    print(f"\n📄 Dataset classes: {config['names']}")
    print(f"   Number of classes: {config['nc']}")

✅ Dataset found at: /kaggle/input/datasets/kmanivignesh/smoking-finalised-dataset/Smoking.v3i.yolov8

📄 Dataset classes: ['smoking']
   Number of classes: 1


In [3]:
# ============================================
# 2. CONFIGURATION
# ============================================
CONFIG = {
    'data_yaml': data_yaml,
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'model_name': 'yolov9c.pt',      # YOLOv9c (default model)
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'patience': 10,
    'save_period': 10,
    'project': 'smoking_detection_yolov9',
    'name': 'exp1',
    'pretrained': True,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'weight_decay': 0.0005,
}

print("\n" + "="*60)
print("SMOKING DETECTION - YOLOv9 TRAINING PIPELINE")
print("="*60)
print(f"Device: {CONFIG['device']}")
print(f"Model: {CONFIG['model_name']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Data YAML: {CONFIG['data_yaml']}")


SMOKING DETECTION - YOLOv9 TRAINING PIPELINE
Device: 0
Model: yolov9c.pt
Epochs: 100
Data YAML: /kaggle/input/datasets/kmanivignesh/smoking-finalised-dataset/Smoking.v3i.yolov8/data.yaml


In [4]:
# ============================================
# 3. TRAIN THE MODEL
# ============================================
print("\n🚀 Starting YOLOv9 model training...")
model = YOLO(CONFIG['model_name']) if CONFIG['pretrained'] else YOLO('yolov9c.yaml')

results = model.train(
    data=CONFIG['data_yaml'],
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    patience=CONFIG['patience'],
    save_period=CONFIG['save_period'],
    project=CONFIG['project'],
    name=CONFIG['name'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    weight_decay=CONFIG['weight_decay'],
    augment=True,          # Standard augmentations
    mixup=0.2,             # MixUp augmentation
    copy_paste=0.3,        # Copy-Paste augmentation
    seed=42,
    verbose=True,
)

print("\n✅ YOLOv9 Training complete!")


🚀 Starting YOLOv9 model training...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/kmanivignesh/smoking-finalised-dataset/Smoking.v3i.yolov8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov9c.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, 

In [5]:
# ============================================
# 4. VALIDATION
# ============================================
best_model_path = Path(f"{CONFIG['project']}/{CONFIG['name']}/weights/best.pt")
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    val_results = best_model.val(data=CONFIG['data_yaml'])
    
    print("\n📈 YOLOv9 Validation Metrics:")
    print(f"   mAP50: {val_results.box.map50:.4f}")
    print(f"   mAP50-95: {val_results.box.map:.4f}")
    print(f"   Precision: {val_results.box.mp:.4f}")
    print(f"   Recall: {val_results.box.mr:.4f}")
else:
    print("⚠️ best.pt not found. Skipping validation.")

⚠️ best.pt not found. Skipping validation.


In [6]:
# ============================================
# 5. EXTRACT & PLOT TRAINING HISTORY
# ============================================
results_csv = Path(f"{CONFIG['project']}/{CONFIG['name']}/results.csv")
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    # Rename columns for clarity
    rename_map = {}
    for col in df.columns:
        if 'epoch' in col.lower():
            rename_map[col] = 'epoch'
        elif 'train/box_loss' in col:
            rename_map[col] = 'train_box_loss'
        elif 'train/cls_loss' in col:
            rename_map[col] = 'train_cls_loss'
        elif 'metrics/precision' in col:
            rename_map[col] = 'precision'
        elif 'metrics/recall' in col:
            rename_map[col] = 'recall'
        elif 'metrics/mAP50(B)' in col or 'metrics/mAP50' in col:
            rename_map[col] = 'mAP50'
        elif 'metrics/mAP50-95(B)' in col or 'metrics/mAP50-95' in col:
            rename_map[col] = 'mAP50_95'
        elif 'val/box_loss' in col:
            rename_map[col] = 'val_box_loss'
        elif 'val/cls_loss' in col:
            rename_map[col] = 'val_cls_loss'
        elif 'lr/pg0' in col:
            rename_map[col] = 'learning_rate'
    
    df = df.rename(columns=rename_map)
    
    # Find best epoch
    if 'mAP50' in df.columns:
        best_idx = df['mAP50'].idxmax()
        best_epoch = df.loc[best_idx, 'epoch']
        best_map50 = df.loc[best_idx, 'mAP50']
        print(f"\n🏆 Best YOLOv9 Model: Epoch {int(best_epoch)} with mAP50 = {best_map50:.4f}")
    
    # Plot training curves
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('YOLOv9 Smoking Detection Training', fontsize=14, fontweight='bold')
    
    # Box Loss
    if 'train_box_loss' in df.columns:
        axes[0,0].plot(df['epoch'], df['train_box_loss'], label='Train Box Loss')
    if 'val_box_loss' in df.columns:
        axes[0,0].plot(df['epoch'], df['val_box_loss'], label='Val Box Loss')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Box Loss')
    axes[0,0].set_title('Box Regression Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Classification Loss
    if 'train_cls_loss' in df.columns:
        axes[0,1].plot(df['epoch'], df['train_cls_loss'], label='Train Cls Loss')
    if 'val_cls_loss' in df.columns:
        axes[0,1].plot(df['epoch'], df['val_cls_loss'], label='Val Cls Loss')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('Cls Loss')
    axes[0,1].set_title('Classification Loss')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Precision & Recall
    if 'precision' in df.columns:
        axes[1,0].plot(df['epoch'], df['precision'], label='Precision', color='green')
    if 'recall' in df.columns:
        axes[1,0].plot(df['epoch'], df['recall'], label='Recall', color='orange')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('Score')
    axes[1,0].set_title('Precision & Recall')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # mAP
    if 'mAP50' in df.columns:
        axes[1,1].plot(df['epoch'], df['mAP50'], label='mAP50', color='blue')
    if 'mAP50_95' in df.columns:
        axes[1,1].plot(df['epoch'], df['mAP50_95'], label='mAP50-95', color='red')
    axes[1,1].set_xlabel('Epoch')
    axes[1,1].set_ylabel('mAP')
    axes[1,1].set_title('Mean Average Precision')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display YOLOv9's saved plots
    for plot_name in ['confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
        plot_path = Path(f"{CONFIG['project']}/{CONFIG['name']}/{plot_name}")
        if plot_path.exists():
            from PIL import Image
            plt.figure(figsize=(8,8))
            plt.imshow(Image.open(plot_path))
            plt.title(plot_name.replace('_', ' ').replace('.png', '').title())
            plt.axis('off')
            plt.show()

In [7]:
# ============================================
# 6. INFERENCE ON TEST IMAGES
# ============================================
test_images_dir = Path(base_path) / "test" / "images"
if test_images_dir.exists() and best_model_path.exists():
    print("\n🔍 Running YOLOv9 inference on test images...")
    test_model = YOLO(str(best_model_path))
    test_images = list(test_images_dir.glob("*.*"))[:6]
    
    if test_images:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        for i, img_path in enumerate(test_images):
            results = test_model(str(img_path), conf=0.25)
            img_with_boxes = results[0].plot()
            axes[i].imshow(img_with_boxes)
            axes[i].set_title(img_path.name, fontsize=8)
            axes[i].axis('off')
        plt.suptitle("YOLOv9 Smoking Detection on Test Images", fontsize=14)
        plt.tight_layout()
        plt.show()